# EXP-016 dev-small 확인: 점수 기반 결합에서 조건 목록 제외

EXP-014(RRF k 20), EXP-015(점수 결합 + 조건 목록 RRF), EXP-016(점수 결합, 조건 목록 대신 결합 목록 밖 최다 일치 문서를 뒤에 붙임)의 dev-small 지표, EXP-015 대비 순위가 바뀐 질의, 결합 목록 밖 최다 일치 문서가 붙은 경우를 본다(`docs/experiments/EXP-016.md`).

In [1]:
from pathlib import Path

import pandas as pd

from rag.conditions import build_extractor
from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
RUN_FILES = {'EXP-014': f'{PREFIX}-k20-dev-small.csv',
             'EXP-015': f'{PREFIX}-cc-dev-small.csv',
             'EXP-016': f'{PREFIX}-cc-nocl-dev-small.csv'}

## 지표

In [3]:
gold = load_gold(load_cfg(), 'dev-small')
runs = {name: read_run(RUNS / f) for name, f in RUN_FILES.items()}
ranks = {name: gold_ranks(run, gold, 10) for name, run in runs.items()}
print(len(gold), 'queries')
table = pd.DataFrame({n: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                      for n, r in ranks.items()})
print(table.round(4))

1039 queries
           EXP-014  EXP-015  EXP-016
recall@1    0.9259   0.9384   0.9384
recall@5    0.9875   0.9885   0.9885
recall@10   0.9904   0.9923   0.9923


## EXP-015 대비 질의별 이동

In [4]:
b, c = ranks['EXP-015'], ranks['EXP-016']
for k in (1, 5, 10):
    gain = int(((b.isna() | (b > k)) & (c <= k)).sum())
    loss = int(((b <= k) & (c.isna() | (c > k))).sum())
    print(f'top{k}: 얻음 {gain}, 잃음 {loss}, 순증 {gain - loss}')
top10 = {n: r.sort_values(['qid', 'rank']).groupby('qid')['doc_id'].apply(list)
         for n, r in runs.items()}
changed = {q for q in top10['EXP-015'].index if top10['EXP-015'][q] != top10['EXP-016'][q]}
print('상위 10개 목록이 바뀐 질의', len(changed))

top1: 얻음 0, 잃음 0, 순증 0
top5: 얻음 0, 잃음 0, 순증 0
top10: 얻음 0, 잃음 0, 순증 0
상위 10개 목록이 바뀐 질의 0


## 조건 질의와 결합 목록 밖 최다 일치 문서

EXP-016 설정으로 질의마다 목록을 다시 만든다. 조건 없이 만든 결합 목록과 비교해 뒤에 붙은 문서를 찾는다. 최다 일치 문서가 있는 질의(EXP-015에서 조건 목록이 생긴 질의), 붙은 문서가 있는 질의, 붙은 문서가 정답인 질의를 센다.

In [6]:
cfg = load_cfg(['+exp=exp016', 'index.scope=dev-small'])
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = read_texts(text_path, qids)
emb = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
tokens = ret.query_tokens(texts)
ex = build_extractor(cfg)
rows = []
for j, q in enumerate(qids):
    row = ret.chunk_emb @ emb[j]
    named, matched, _ = ret.lists(row, tokens[j], ex.extract(texts[j]))
    base = ret.lists(row, tokens[j], {})[0]['cc']
    added = named['cc'][len(base):]
    rows.append({'qid': q, 'cond': bool(matched), 'n_matched': len(matched),
                 'n_added': len(added), 'gold_added': gold.loc[q, 'doc_id'] in added})
stat = pd.DataFrame(rows).set_index('qid')
print('조건 질의', int(stat['cond'].sum()), '/', len(stat))
print('문서가 붙은 질의', int((stat['n_added'] > 0).sum()),
      ', 붙은 문서 합', int(stat['n_added'].sum()))
print('붙은 문서가 정답인 질의', int(stat['gold_added'].sum()))
print('순위가 바뀐 질의 중 조건 질의가 아닌 것', len(changed - set(stat.index[stat['cond']])))

조건 질의 0 / 1039
문서가 붙은 질의 0 , 붙은 문서 합 0
붙은 문서가 정답인 질의 0
순위가 바뀐 질의 중 조건 질의가 아닌 것 0
